# Foundation experiments

Один ноутбук для всех конфигураций. GPU T4 ×2, интернет выключен. Подключите данные соревнования и private dataset проекта. Поддерживаются распакованные `package.json`, `src`, `configs`, `inputs` и сохранённый `base.zip`.

Сначала выполните три контроля. Затем меняйте только SELECTED. Все 21 конфигурации за один 50-минутный запуск выполнять не нужно. Время каждой измеряется. Это исследовательский ноутбук для train/val, а не финальный train+val сабмит.

In [ ]:
SELECTED = ["019_base", "020_resnet", "029_siglip"]
SEEDS = None  # None uses each configuration seed; [421, 853] repeats selected runs.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch

assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
input_root = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("KAGGLE_WORK_ROOT", "/kaggle/working"))
projects = sorted({p.parent for p in input_root.rglob("package.json")
                   if all((p.parent / name).is_dir() for name in ("src", "configs", "inputs"))})
assert len(projects) <= 1, f"Several unpacked project datasets found: {projects}"
if projects:
    # Kaggle exposes uploaded ZIP contents as a read-only dataset directory.
    # Code, configurations and inputs can be read directly from that directory.
    project = projects[0]
else:
    archives = list(input_root.rglob("base.zip"))
    assert len(archives) == 1, "Attach one project dataset with package.json, src, configs, inputs or base.zip"
    project = work_root / "project"
    project.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as bundle:
        for entry in bundle.infolist():
            dest = (project / entry.filename).resolve()
            assert dest.is_relative_to(project.resolve()), "Unsafe archive path"
        bundle.extractall(project)

manifest = json.loads((project / "package.json").read_text(encoding="utf-8"))
assert manifest.get("version") == 1, "Unsupported package manifest version"
for name, digest in manifest["files"].items():
    path = (project / name).resolve()
    assert path.is_relative_to(project.resolve()), "Unsafe manifest path"
    with path.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, f"Package checksum mismatch: {name}"
sys.path.insert(0, str(project))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
roots = sorted({p.parent for p in input_root.rglob("train.csv")
                if (p.parent / "val.csv").is_file()
                and (p.parent / "train").is_dir() and (p.parent / "val").is_dir()})
assert len(roots) == 1, f"Competition CSVs and train/val image directories must share one root: {roots}"
data = roots[0]
print("Project:", project, "Data:", data)


Все модели, цели и PCA обучаются только на train. В каждой новой конфигурации вторая ветка смеси фиксируется на историческом контроле. Результаты и веса записываются в `artefacts`, временные файлы — в `/kaggle/temp/foundation`.

In [ ]:
from src.suite import run_suite
results = run_suite(project=project, data=data, inputs=project / "inputs",
                    selected=SELECTED, seeds=SEEDS,
                    work=Path("/kaggle/temp/foundation"),
                    output=Path("/kaggle/working/artefacts"))
results
